In [342]:
import numpy as np
import torch
import tensorflow as tf

In [343]:
tf.test.is_gpu_available()

2024-04-21 04:31:25.601950: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-04-21 04:31:25.602095: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-04-21 04:31:25.602142: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-

True

L355
2024-04-21 04:31:25.602256: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2024-04-21 04:31:25.602300: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /device:GPU:0 with 17706 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:01:00.0, compute capability: 8.6


In [344]:
PER_FRAME_FEATURE = 1629
def extractPoseHand(npy):
    np_matrix=npy.reshape(-1,PER_FRAME_FEATURE)

    pose  = np_matrix[: ,0:98]
    hands  = np_matrix[: ,1503:PER_FRAME_FEATURE]
    
 
    arr = np.concatenate((pose, hands), axis=1)


    return arr

In [345]:
def getNPY_224(split, name):
   file_name =  f"../relativeQ/BdSLW60/{split}/{name}" 
    
   npy = np.load(file_name)
   npy = extractPoseHand(npy)
   return torch.from_numpy(npy)


In [346]:
import pickle
import gzip
import os

# dict_keys(['name', 'signer', 'gloss', 'text', 'sign'])

# Test
test_path  = "../relativeQ/BdSLW60/Test"
annotations = []   

for trial in os.listdir(test_path):
    sample = {}

    sample['name'] = "Test/"+trial
    sample['signer'] = trial.split("_")[1]
    sample['gloss'] = trial.split("_")[0]
    sample['text'] = trial.split("_")[0]
    sample['sign'] = getNPY_224("Test", trial)

    annotations.append(sample)


In [347]:
# save the gzipped file
with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.test", "wb") as file:
    pickle.dump(annotations, file)
file.close()

In [348]:
VALIDATION_USER ="U18"

In [349]:
# Dev
train_path  = "../relativeQ/BdSLW60/Training"
annotations_dev = []   
annotations_train = []   

for trial in os.listdir(train_path):
    sample = {}
    
    sample['signer'] = trial.split("_")[1]
    sample['gloss'] = trial.split("_")[0]
    sample['text'] = trial.split("_")[0]
    sample['sign'] = getNPY_224("Training", trial)

    user = trial.split("_")[1]
    if user == VALIDATION_USER:
        sample['name'] = "Train/"+trial
        annotations_dev.append(sample)
    else:
        sample['name'] = "Train/"+trial
        annotations_train.append(sample)

In [350]:
  
import random
train_path  = "../relativeQ/BdSLW60/Training"
fileList = os.listdir(train_path)
fileList.sort()



p_list = []

all_train = []
all_dev = []

previous_trial = fileList[0]
s = previous_trial.split("_")
p_prefix = s[0]+"_"+s[1]



for index in range(1, len(fileList)):
   
    p_list.append(previous_trial)
    

    next_trial = fileList[index]
    s = next_trial.split("_")
    next_prefix = s[0]+"_"+s[1]

    if next_prefix != p_prefix or index == len(fileList)-1:
        if index == len(fileList)-1:
            p_list.append(next_trial)
        # do process
        if len(p_list) != 0:
           random.shuffle(p_list)

           size = len(p_list)
           dev_size = int(size * 0.1)

           dev_set = p_list[:dev_size]
           train_set = p_list[dev_size:]
           total_size = len(dev_set) + len(train_set)
           assert  total_size == size, f"{len(dev_set)}  {len(train_set)}  {size}"

           all_train.extend(train_set)
           all_dev.extend(dev_set)
           p_list = []

        # else:
        #     print("Empty list")

    previous_trial = next_trial
    p_prefix = next_prefix


In [351]:
# import pickle
# import gzip
# import os

# # Train

# annotations = []   
# for trial in all_train:
#     sample = {}

#     sample['name'] = "Train/"+trial
#     sample['signer'] = trial.split("_")[1]
#     sample['gloss'] = trial.split("_")[0]
#     sample['text'] = trial.split("_")[0]
#     sample['sign'] = getNPY_224("Training", trial)

#     annotations.append(sample)
# # save the gzipped file
# with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.UBS.train", "wb") as file:
#     pickle.dump(annotations, file)
# file.close()

In [352]:
# import pickle
# import gzip
# import os

# # Dev

# annotations = []   
# for trial in all_dev:
#     sample = {}

#     sample['name'] = "Train/"+trial
#     sample['signer'] = trial.split("_")[1]
#     sample['gloss'] = trial.split("_")[0]
#     sample['text'] = trial.split("_")[0]
#     sample['sign'] = getNPY_224("Training", trial)

#     annotations.append(sample)
# # save the gzipped file
# with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.UBS.dev", "wb") as file:
#     pickle.dump(annotations, file)
# file.close()

In [353]:
# save the gzipped file
with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.train", "wb") as file:
    pickle.dump(annotations_train, file)
file.close()

In [354]:
# save the gzipped file
with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.dev", "wb") as file:
    pickle.dump(annotations_dev, file)
file.close()

In [355]:
# check sanity
i = 0
with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.UBS.train", "rb") as file:
    data_test = pickle.load(file)
    print(len(data_test))

file.close()

7404


In [356]:
# check sanity
i = 0
with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.UBS.dev", "rb") as file:
    data_test = pickle.load(file)
    print(len(data_test))

file.close()

627


In [357]:
# check sanity
i = 0
with gzip.open("../gzip_dataset/BdSLW60/BdSLW60_224.dev", "rb") as file:
    data_test = pickle.load(file)
    print(len(data_test))
    for trial in data_test:
        print(trial['sign'].shape)
        break

file.close()

141
torch.Size([165, 224])
